# Experimento de Comparação de Janelas de Treinamento

**Projeto:** Renewable Energy MLOps — Wind Power Forecasting  
**Status:** benchmark concluído e congelado  
**Branch:** `experiment/training-window-comparison`  
**Snapshot temporal:** congelado  
**Arquitetura:** `LGBM + XGBoost + Random Forest`  
**OOT:** setembro/2026

Este notebook documenta o benchmark controlado utilizado para avaliar
quanto histórico deve ser utilizado no treinamento do Temporal Stacking.

A variável experimental foi **somente a janela temporal de treinamento**:

```text
EXPANDING
└── todo o histórico canônico disponível antes do OOT

ROLLING 24M
└── últimos 24 meses antes do OOT

ROLLING 12M
└── últimos 12 meses antes do OOT
```

Os três tratamentos foram executados separadamente sob o mesmo
protocolo experimental. As runs finais estão congeladas neste notebook.

O notebook anterior de simplificação do ensemble permanece congelado.
Este experimento não alterou a arquitetura do modelo e não executou o
Quality Gate operacional.


## 1. Objetivo

O objetivo é verificar se restringir o treinamento aos regimes mais
recentes melhora ou preserva a capacidade de generalização em relação
à expanding window.

A expanding window não é tratada como proteção automática contra
data drift. À medida que o processo gerador dos dados muda, histórico
mais antigo pode deixar de representar adequadamente o regime atual.

O benchmark mantém fixos:

- o mesmo snapshot Gold;
- o mesmo OOT de setembro/2026;
- as mesmas 9 features;
- a mesma arquitetura `LGBM + XGB + RF`;
- o mesmo `TimeSeriesSplit` externo;
- o mesmo `TimeSeriesSplit` interno do Temporal Stacking;
- 20 trials Optuna por tratamento;
- nMAE (%) em MW como métrica primária de otimização.

Somente o **limite inferior do TRAIN** muda entre os tratamentos.


## 2. Hipóteses

### Pergunta experimental

Quanto histórico deve ser utilizado para treinar o Temporal Stacking
quando o futuro avaliado é setembro/2026?

### Tratamentos

```text
EXPANDING
2024-03-21 00:00 → 2026-09-01 02:00

ROLLING 24M
2024-09-01 03:00 → 2026-09-01 02:00

ROLLING 12M
2025-09-01 03:00 → 2026-09-01 02:00

OOT COMUM
2026-09-01 03:00 → 2026-10-01 02:00
```

### Hipótese de trabalho

Uma janela recente pode reduzir a influência de regimes históricos
menos representativos do estado atual do sistema.

O experimento não assume antecipadamente que uma rolling window será
superior. A expanding window permanece como baseline.


## 3. Contrato experimental

```text
Snapshot Gold congelado
        ↓
split_training_window_snapshot()
        │
        ├── None → expanding
        ├── 24   → rolling 24m
        └── 12   → rolling 12m
        │
        └───────────────┐
                        ↓
                   TRAIN distinto
                        +
                    OOT idêntico
                        ↓
                 Optuna idêntico
                        ↓
             LGBM + XGB + RF
                        ↓
                     MLflow
```

O notebook reutiliza a função oficial de split do `training_flow`.
Nenhuma segunda implementação de corte temporal é criada aqui.

O `continuous_training_pipeline()` não é chamado, portanto
`evaluate_and_promote()` também não é executado.


In [24]:
import importlib
import inspect
import json

import pandas as pd
from mlflow.tracking import MlflowClient

import energy_mlops.models.optimize_stacking_ensemble as optimization_module
import energy_mlops.models.train_stacking_ensemble as training_module
from energy_mlops.config import settings
from energy_mlops.data.feature_utils import get_model_feature_columns
from energy_mlops.data.snapshot_validation import audit_gold_snapshot
from energy_mlops.models.ensemble_config import validate_enabled_estimators
from energy_mlops.pipelines.training_flow import (
    build_training_dataset_labels,
    persist_training_datasets_task,
    split_training_window_snapshot,
)

optimization_module = importlib.reload(
    optimization_module
)

training_module = importlib.reload(
    training_module
)

print("✅ Módulos recarregados.")

print(
    "Optimizer:",
    inspect.signature(
        optimization_module.run_optimization
    ),
)

print(
    "Trainer:",
    inspect.signature(
        training_module.train_stacking_regressor
    ),
)


✅ Módulos recarregados.
Optimizer: (df_train: pandas.DataFrame, train_file: str, n_trials: int = 20, n_splits: int = 3, stacking_n_splits: int = 5, enabled_estimators: tuple[str, ...] = ('lgbm', 'xgboost', 'rf')) -> energy_mlops.models.interfaces.OptimizationResult
Trainer: (df_train: pandas.DataFrame, df_test: pandas.DataFrame, train_file: str, test_file: str, best_params: dict | None = None, optimization_run_id: str | None = None, enabled_estimators: tuple[str, ...] = ('lgbm', 'xgboost', 'rf')) -> tuple[str, float]


## 4. Configuração do benchmark

A arquitetura permanece fixa. O único fator experimental é a janela.

`RUN_EXPERIMENTS=False` impede iniciar três otimizações longas por
acidente ao abrir ou executar o notebook.

`PERSIST_DERIVED_DATASETS=False` impede escrita acidental dos artefatos
TRAIN/OOT no Gold.


In [25]:
EXPERIMENT_CONFIG = {
    "n_trials": 20,
    "n_splits": 3,
    "stacking_n_splits": 5,
}

TRAINING_WINDOWS = {
    "Expanding": None,
    "Rolling 24m": 24,
    "Rolling 12m": 12,
}

BENCHMARK_SEQUENCE = (
    "Expanding",
    "Rolling 24m",
    "Rolling 12m",
)

ENABLED_ESTIMATORS = (
    validate_enabled_estimators(
        (
            "lgbm",
            "xgboost",
            "rf",
        )
    )
)

# Benchmark concluído: não executar novas otimizações por padrão.
WINDOW_TO_RUN = "Rolling 12m"

RUN_EXPERIMENTS = False
PERSIST_DERIVED_DATASETS = False

assert WINDOW_TO_RUN in TRAINING_WINDOWS

(
    EXPERIMENT_CONFIG,
    TRAINING_WINDOWS,
    BENCHMARK_SEQUENCE,
    ENABLED_ESTIMATORS,
    WINDOW_TO_RUN,
)


({'n_trials': 20, 'n_splits': 3, 'stacking_n_splits': 5},
 {'Expanding': None, 'Rolling 24m': 24, 'Rolling 12m': 12},
 ('Expanding', 'Rolling 24m', 'Rolling 12m'),
 ('lgbm', 'xgboost', 'rf'),
 'Rolling 12m')

## 5. Snapshot temporal congelado e proveniência

O benchmark utiliza o mesmo snapshot Gold validado anteriormente:

```text
GOLD
arquivo: dataset_renewable_energy_2024_03_2026_09.parquet
rows: 22,145
range: 2024-03-21 00:00:00 → 2026-10-01 02:00:00
missing timestamps: 34
gaps: 2
features: 9

OOT — setembro/2026
rows: 720
range: 2026-09-01 03:00:00 → 2026-10-01 02:00:00
missing timestamps: 0
gaps: 0
```

A capacidade instalada segue o contrato canônico
ABEEólica/INFOVENTO. O OOT é mantido idêntico em todos os tratamentos.


In [26]:
GOLD_FILE = (
    "dataset_renewable_energy_2024_03_2026_09.parquet"
)

GOLD_PATH = (
    f"s3://{settings.RUSTFS_BUCKET}/gold/"
    f"{GOLD_FILE}"
)

OOT_YEAR = 2026
OOT_MONTH = 9

EXPECTED_GOLD_ROWS = 22_145
EXPECTED_OOT_ROWS = 720

EXPECTED_GOLD_START = pd.Timestamp(
    "2024-03-21 00:00:00"
)
EXPECTED_GOLD_END = pd.Timestamp(
    "2026-10-01 02:00:00"
)
EXPECTED_TRAIN_END = pd.Timestamp(
    "2026-09-01 02:00:00"
)
EXPECTED_OOT_START = pd.Timestamp(
    "2026-09-01 03:00:00"
)
EXPECTED_OOT_END = pd.Timestamp(
    "2026-10-01 02:00:00"
)

EXPECTED_WINDOWS = {
    "Expanding": {
        "months": None,
        "rows": 21_425,
        "start": pd.Timestamp(
            "2024-03-21 00:00:00"
        ),
        "missing_timestamps": 34,
        "gaps": 2,
    },
    "Rolling 24m": {
        "months": 24,
        "rows": 17_486,
        "start": pd.Timestamp(
            "2024-09-01 03:00:00"
        ),
        "missing_timestamps": 34,
        "gaps": 2,
    },
    "Rolling 12m": {
        "months": 12,
        "rows": 8_760,
        "start": pd.Timestamp(
            "2025-09-01 03:00:00"
        ),
        "missing_timestamps": 0,
        "gaps": 0,
    },
}


In [27]:
df_gold = pd.read_parquet(
    GOLD_PATH,
    storage_options=settings.storage_options,
)

gold_audit = audit_gold_snapshot(
    df_gold
)

MODEL_FEATURES = (
    get_model_feature_columns()
)

window_datasets = {}

for label, months in TRAINING_WINDOWS.items():
    df_train, df_oot = (
        split_training_window_snapshot(
            df_gold,
            oot_year=OOT_YEAR,
            oot_month=OOT_MONTH,
            training_window_months=months,
        )
    )

    train_audit = audit_gold_snapshot(
        df_train
    )
    oot_audit = audit_gold_snapshot(
        df_oot
    )

    train_file, oot_file = (
        build_training_dataset_labels(
            df_train,
            oot_year=OOT_YEAR,
            oot_month=OOT_MONTH,
            training_window_months=months,
        )
    )

    window_datasets[label] = {
        "months": months,
        "train": df_train,
        "oot": df_oot,
        "train_audit": train_audit,
        "oot_audit": oot_audit,
        "train_file": train_file,
        "oot_file": oot_file,
    }

print("✅ Janelas derivadas do snapshot Gold.")


✅ Janelas derivadas do snapshot Gold.


In [28]:
snapshot_rows = []

for label, payload in window_datasets.items():
    df_train = payload["train"]
    df_oot = payload["oot"]

    snapshot_rows.append(
        {
            "Janela": label,
            "Meses": payload["months"],
            "Train rows": len(df_train),
            "Train início": df_train["date"].min(),
            "Train fim": df_train["date"].max(),
            "Train gaps": (
                payload["train_audit"].gap_count
            ),
            "Train missing": (
                payload[
                    "train_audit"
                ].missing_timestamps
            ),
            "OOT rows": len(df_oot),
            "OOT início": df_oot["date"].min(),
            "OOT fim": df_oot["date"].max(),
            "TRAIN file": payload["train_file"],
            "OOT file": payload["oot_file"],
        }
    )

snapshot_summary = (
    pd.DataFrame(snapshot_rows)
    .set_index("Janela")
)

snapshot_summary


,Meses,Train rows,Train início,Train fim,Train gaps,Train missing,OOT rows,OOT início,OOT fim,TRAIN file,OOT file
Janela,,,,,,,,,,,
Expanding,NaN,21425,2024-03-21 00:00:00,2026-09-01 02:00:00,2,34,720,2026-09-01 03:00:00,2026-10-01 02:00:00,train_wind_energy_2024_03_expanding_up_to_2026...,oot_test_wind_energy_2026_09.parquet
Rolling 24m,24.0,17486,2024-09-01 03:00:00,2026-09-01 02:00:00,2,34,720,2026-09-01 03:00:00,2026-10-01 02:00:00,train_wind_energy_2024_09_rolling_24m_up_to_20...,oot_test_wind_energy_2026_09.parquet
Rolling 12m,12.0,8760,2025-09-01 03:00:00,2026-09-01 02:00:00,0,0,720,2026-09-01 03:00:00,2026-10-01 02:00:00,train_wind_energy_2025_09_rolling_12m_up_to_20...,oot_test_wind_energy_2026_09.parquet


## 6. Sanity checks antes da execução

O benchmark só deve rodar se:

1. o Gold permanecer exatamente no snapshot congelado;
2. as três janelas tiverem os limites temporais esperados;
3. o OOT for exatamente o mesmo DataFrame nos três tratamentos;
4. TRAIN e OOT não tiverem overlap;
5. o OOT estiver completo e sem gaps;
6. os gaps históricos permanecerem somente nas janelas que os incluem;
7. o contrato do modelo continuar com exatamente 9 features;
8. a arquitetura permanecer `LGBM + XGB + RF`;
9. o orçamento Optuna permanecer idêntico;
10. nenhuma execução utilizar o OOT durante a otimização.


In [29]:
def validate_experiment_inputs():
    assert len(df_gold) == EXPECTED_GOLD_ROWS

    assert (
        pd.Timestamp(
            df_gold["date"].min()
        )
        == EXPECTED_GOLD_START
    )

    assert (
        pd.Timestamp(
            df_gold["date"].max()
        )
        == EXPECTED_GOLD_END
    )

    assert gold_audit.missing_timestamps == 34
    assert gold_audit.gap_count == 2

    assert len(MODEL_FEATURES) == 9

    assert ENABLED_ESTIMATORS == (
        "lgbm",
        "xgboost",
        "rf",
    )

    assert EXPERIMENT_CONFIG == {
        "n_trials": 20,
        "n_splits": 3,
        "stacking_n_splits": 5,
    }

    required_columns = {
        "target_fc",
        "wind_generation_mw",
        "capacidade_mw",
        *MODEL_FEATURES,
    }

    reference_oot = None

    for label, expected in (
        EXPECTED_WINDOWS.items()
    ):
        payload = window_datasets[label]

        assert (
            payload["months"]
            == expected["months"]
        )

        df_train = payload["train"]
        df_oot = payload["oot"]

        train_audit = payload[
            "train_audit"
        ]
        oot_audit = payload[
            "oot_audit"
        ]

        assert len(df_train) == expected["rows"]

        assert (
            pd.Timestamp(
                df_train["date"].min()
            )
            == expected["start"]
        )

        assert (
            pd.Timestamp(
                df_train["date"].max()
            )
            == EXPECTED_TRAIN_END
        )

        assert len(df_oot) == EXPECTED_OOT_ROWS

        assert (
            pd.Timestamp(
                df_oot["date"].min()
            )
            == EXPECTED_OOT_START
        )

        assert (
            pd.Timestamp(
                df_oot["date"].max()
            )
            == EXPECTED_OOT_END
        )

        assert (
            df_train["date"].max()
            < df_oot["date"].min()
        )

        assert (
            train_audit.missing_timestamps
            == expected[
                "missing_timestamps"
            ]
        )

        assert (
            train_audit.gap_count
            == expected["gaps"]
        )

        assert (
            oot_audit.missing_timestamps
            == 0
        )
        assert oot_audit.gap_count == 0

        missing_train = (
            required_columns
            - set(df_train.columns)
        )
        assert not missing_train, (
            f"{label}: colunas ausentes no TRAIN: "
            f"{sorted(missing_train)}"
        )

        missing_oot = (
            required_columns
            - set(df_oot.columns)
        )
        assert not missing_oot, (
            f"{label}: colunas ausentes no OOT: "
            f"{sorted(missing_oot)}"
        )

        if reference_oot is None:
            reference_oot = df_oot
        else:
            pd.testing.assert_frame_equal(
                reference_oot,
                df_oot,
            )

    print("✅ Snapshot Gold validado.")
    print("✅ Expanding / 24m / 12m validadas.")
    print("✅ OOT idêntico nos três tratamentos.")
    print("✅ TRAIN/OOT sem overlap.")
    print("✅ Contrato de 9 features preservado.")
    print("✅ Arquitetura LGBM + XGB + RF congelada.")
    print("✅ Benchmark pronto para execução.")


validate_experiment_inputs()


✅ Snapshot Gold validado.
✅ Expanding / 24m / 12m validadas.
✅ OOT idêntico nos três tratamentos.
✅ TRAIN/OOT sem overlap.
✅ Contrato de 9 features preservado.
✅ Arquitetura LGBM + XGB + RF congelada.
✅ Benchmark pronto para execução.


## 7. Persistência opcional dos artefatos derivados

O optimizer e o trainer registram no MLflow os nomes dos arquivos
utilizados na linhagem.

Antes da primeira execução, os três TRAIN derivados podem ser
materializados no Gold. O arquivo OOT possui o mesmo conteúdo e o mesmo
nome nos três tratamentos.

A flag permanece `False` por segurança.


In [ ]:
if PERSIST_DERIVED_DATASETS:
    for label, payload in (
        window_datasets.items()
    ):
        (
            persisted_train_file,
            persisted_oot_file,
        ) = persist_training_datasets_task.fn(
            payload["train"],
            payload["oot"],
            oot_year=OOT_YEAR,
            oot_month=OOT_MONTH,
            training_window_months=(
                payload["months"]
            ),
        )

        assert (
            persisted_train_file
            == payload["train_file"]
        )

        assert (
            persisted_oot_file
            == payload["oot_file"]
        )

        print(
            f"✅ {label}: "
            f"{persisted_train_file}"
        )

    print(
        "✅ Artefatos derivados persistidos."
    )
else:
    print(
        "ℹ️ Persistência desabilitada. "
        "Defina PERSIST_DERIVED_DATASETS=True "
        "somente quando quiser materializar "
        "os artefatos derivados."
    )


## 8. Governança do benchmark

Este notebook chama diretamente:

```text
run_optimization()
        ↓
train_stacking_regressor()
        ↓
MLflow
```

Ele **não chama**:

```text
continuous_training_pipeline()
evaluate_and_promote()
```

Portanto, nenhuma run experimental pode alterar automaticamente o
alias `@champion`.

Dependendo do comportamento atual do trainer, versões experimentais
podem ainda ser registradas no Model Registry. Isso é aceitável para
rastreabilidade desde que nenhuma promoção automática seja executada.

As runs recebem tags explícitas com a janela utilizada.


In [ ]:
mlflow_client = MlflowClient(
    tracking_uri=settings.MLFLOW_TRACKING_URI
)


def assert_mlflow_available():
    """
    Falha antes de iniciar uma otimização longa caso o
    endpoint MLflow não esteja acessível.
    """

    try:
        mlflow_client.search_experiments(
            max_results=1
        )
    except Exception as exc:
        raise RuntimeError(
            "MLflow indisponível antes da execução. "
            "Verifique o port-forward em localhost:5000."
        ) from exc

    print("✅ MLflow acessível.")


def tag_window_run(
    run_id: str,
    *,
    label: str,
    months: int | None,
    df_train: pd.DataFrame,
    df_oot: pd.DataFrame,
    run_role: str,
):
    strategy = (
        "expanding"
        if months is None
        else "rolling"
    )

    tags = {
        "experiment_family": (
            "training_window_comparison"
        ),
        "experiment_variable": (
            "training_window"
        ),
        "benchmark_label": label,
        "run_role": run_role,
        "training_window_strategy": strategy,
        "training_window_months": (
            "all_available"
            if months is None
            else str(months)
        ),
        "training_start": str(
            df_train["date"].min()
        ),
        "training_end": str(
            df_train["date"].max()
        ),
        "oot_start": str(
            df_oot["date"].min()
        ),
        "oot_end": str(
            df_oot["date"].max()
        ),
        "fixed_architecture": (
            "lgbm,xgboost,rf"
        ),
        "quality_gate_executed": "false",
    }

    for key, value in tags.items():
        mlflow_client.set_tag(
            run_id,
            key,
            value,
        )


## 9. Execução resiliente — benchmark completo em três etapas

O benchmark final será executado novamente desde o início.

Cada tratamento recebe uma otimização Optuna independente e é
executado **isoladamente**, na seguinte ordem:

```text
1. Expanding
2. Rolling 24m
3. Rolling 12m
```

Essa estratégia preserva um único benchmark coerente, mas evita que
uma falha transitória de infraestrutura em um tratamento obrigue a
repetir os tratamentos que já foram concluídos.

Antes de iniciar uma execução longa, mantenha um port-forward resiliente
do MLflow em outro terminal:

```bash
pkill -f 'kubectl port-forward svc/mlflow 5000:5000' || true

while true; do
    kubectl port-forward svc/mlflow 5000:5000
    echo "MLflow port-forward caiu. Reiniciando em 2s..."
    sleep 2
done
```

A execução é controlada por:

```python
WINDOW_TO_RUN = "Expanding"
RUN_EXPERIMENTS = False
```

Para cada tratamento:

1. selecione a janela em `WINDOW_TO_RUN`;
2. defina `RUN_EXPERIMENTS=True`;
3. execute somente a célula de execução;
4. congele imediatamente os Run IDs;
5. volte `RUN_EXPERIMENTS=False`;
6. avance para a próxima janela.

Runs anteriores à nova rodada não são usadas na comparação final.


In [ ]:
def run_window_experiment(
    label: str,
) -> dict:
    """
    Executa exatamente um tratamento do benchmark.
    """

    if label not in window_datasets:
        raise ValueError(
            f"Janela desconhecida: {label!r}. "
            f"Opções: {list(window_datasets)}"
        )

    validate_experiment_inputs()
    assert_mlflow_available()

    payload = window_datasets[label]

    print("\n" + "=" * 72)
    print(f"🚀 Executando: {label}")
    print("=" * 72)

    optimization_result = (
        optimization_module.run_optimization(
            df_train=payload["train"],
            train_file=payload["train_file"],
            n_trials=(
                EXPERIMENT_CONFIG[
                    "n_trials"
                ]
            ),
            n_splits=(
                EXPERIMENT_CONFIG[
                    "n_splits"
                ]
            ),
            stacking_n_splits=(
                EXPERIMENT_CONFIG[
                    "stacking_n_splits"
                ]
            ),
            enabled_estimators=(
                ENABLED_ESTIMATORS
            ),
        )
    )

    optimization_run_id = (
        optimization_result[
            "optimization_run_id"
        ]
    )

    tag_window_run(
        optimization_run_id,
        label=label,
        months=payload["months"],
        df_train=payload["train"],
        df_oot=payload["oot"],
        run_role="optimization",
    )

    (
        training_run_id,
        oot_mae_mw,
    ) = (
        training_module
        .train_stacking_regressor(
            df_train=payload["train"],
            df_test=payload["oot"],
            train_file=payload["train_file"],
            test_file=payload["oot_file"],
            best_params=(
                optimization_result[
                    "best_params"
                ]
            ),
            optimization_run_id=(
                optimization_run_id
            ),
            enabled_estimators=(
                ENABLED_ESTIMATORS
            ),
        )
    )

    tag_window_run(
        training_run_id,
        label=label,
        months=payload["months"],
        df_train=payload["train"],
        df_oot=payload["oot"],
        run_role="training",
    )

    result = {
        "optimization_run_id": (
            optimization_run_id
        ),
        "training_run_id": (
            training_run_id
        ),
        "oot_mae_mw": (
            oot_mae_mw
        ),
    }

    print("\n✅ Tratamento concluído.")
    print(
        json.dumps(
            {
                "window": label,
                **result,
            },
            indent=2,
        )
    )

    return result


experiment_runs = {}

if RUN_EXPERIMENTS:
    experiment_runs[
        WINDOW_TO_RUN
    ] = run_window_experiment(
        WINDOW_TO_RUN
    )
else:
    print(
        "ℹ️ Execução desabilitada. "
        f"Janela selecionada: {WINDOW_TO_RUN!r}."
    )

experiment_runs


## 10. Congelamento das runs finais

As três runs abaixo pertencem à mesma rodada final do benchmark.

O campo de duração do Optuna merece uma distinção importante:

- a duração da **run MLflow de otimização** não representa o tempo total
  do estudo Optuna, porque essa run é criada/logada somente ao final da
  busca;
- por isso, o benchmark final usa os timestamps observados no console:
  criação do study → conclusão do trial 19.

Isso corrige a coluna anteriormente exibida como `Optuna (min)`, que
estava medindo apenas a curta etapa de logging no MLflow.


In [30]:
FROZEN_RUN_IDS = {
    "Expanding": {
        "optimization_run_id": (
            "2fdff0b534104041a01276c0a2fb6009"
        ),
        "training_run_id": (
            "1d13a61244c54f06aa70f43a9993ea37"
        ),
    },
    "Rolling 24m": {
        "optimization_run_id": (
            "b145a449eb6245d4809ff9d3cc3ffc3c"
        ),
        "training_run_id": (
            "a03c85cae21c4d4ebf0fd717d248b775"
        ),
    },
    "Rolling 12m": {
        "optimization_run_id": (
            "6bc964c478be44ca9e4a4cab2d071d08"
        ),
        "training_run_id": (
            "9ce134ad54284ef9aee2cf29db69feb1"
        ),
    },
}

OPTUNA_STUDY_TIMESTAMPS = {
    "Expanding": {
        "start": pd.Timestamp(
            "2026-10-02 11:22:27.650"
        ),
        "end": pd.Timestamp(
            "2026-10-02 11:38:38.930"
        ),
    },
    "Rolling 24m": {
        "start": pd.Timestamp(
            "2026-10-02 11:50:19.227"
        ),
        "end": pd.Timestamp(
            "2026-10-02 12:07:18.536"
        ),
    },
    "Rolling 12m": {
        "start": pd.Timestamp(
            "2026-10-02 12:16:40.539"
        ),
        "end": pd.Timestamp(
            "2026-10-02 12:35:33.485"
        ),
    },
}


def optuna_study_duration_minutes(
    label: str,
) -> float:
    timestamps = (
        OPTUNA_STUDY_TIMESTAMPS[label]
    )

    return (
        timestamps["end"]
        - timestamps["start"]
    ).total_seconds() / 60.0


FROZEN_OPTUNA_STUDY_MINUTES = {
    label: optuna_study_duration_minutes(
        label
    )
    for label in BENCHMARK_SEQUENCE
}

print("✅ Run IDs finais congelados.")
print(
    "✅ Duração real dos estudos Optuna "
    "reconstruída a partir dos timestamps."
)

pd.Series(
    FROZEN_OPTUNA_STUDY_MINUTES,
    name="Optuna study (min)",
).round(4)


✅ Run IDs finais congelados.
✅ Duração real dos estudos Optuna reconstruída a partir dos timestamps.


Expanding      16.1880
Rolling 24m    16.9885
Rolling 12m    18.8824
Name: Optuna study (min), dtype: float64

## 11. Resultados consolidados

A tabela abaixo recupera diretamente das runs MLflow:

- CV nMAE médio e desvio;
- OOT MAE em MW;
- OOT nMAE;
- OOT MAE em fator de capacidade;
- OOT R²;
- pesos OOF do meta-learner;
- duração da run de treinamento;
- limites efetivos do TRAIN.

A duração do estudo Optuna é calculada separadamente a partir dos
timestamps do próprio benchmark, e não da duração da run MLflow de
logging.

Nenhuma equivalência estatística é inferida apenas a partir das
diferenças pontuais.


In [31]:
def run_duration_minutes(run):
    start_time = run.info.start_time
    end_time = run.info.end_time

    if (
        start_time is None
        or end_time is None
    ):
        return None

    return (
        end_time - start_time
    ) / 60_000


def resolve_benchmark_run_ids():
    """
    Parte dos IDs congelados e sobrescreve somente
    uma eventual janela executada na sessão atual.
    """

    resolved = {
        label: values.copy()
        for label, values in (
            FROZEN_RUN_IDS.items()
        )
    }

    for label, values in (
        experiment_runs.items()
    ):
        resolved[label] = {
            "optimization_run_id": (
                values[
                    "optimization_run_id"
                ]
            ),
            "training_run_id": (
                values[
                    "training_run_id"
                ]
            ),
        }

    return resolved


benchmark_run_ids = (
    resolve_benchmark_run_ids()
)

missing_run_ids = [
    label
    for label, values in (
        benchmark_run_ids.items()
    )
    if (
        not values["optimization_run_id"]
        or not values["training_run_id"]
    )
]

if missing_run_ids:
    benchmark_results = None

    print("ℹ️ Benchmark ainda incompleto.")
    print(
        "Tratamentos sem Run IDs congelados:",
        missing_run_ids,
    )

else:
    benchmark_rows = []

    for label, run_ids in (
        benchmark_run_ids.items()
    ):
        optimization_run = (
            mlflow_client.get_run(
                run_ids[
                    "optimization_run_id"
                ]
            )
        )

        training_run = (
            mlflow_client.get_run(
                run_ids[
                    "training_run_id"
                ]
            )
        )

        meta_weights = json.loads(
            training_run.data.tags.get(
                "meta_weights_json",
                "{}",
            )
        )

        payload = window_datasets[
            label
        ]

        benchmark_rows.append(
            {
                "Janela": label,
                "Train rows": len(
                    payload["train"]
                ),
                "Train início": (
                    payload[
                        "train"
                    ]["date"].min()
                ),
                "Train fim": (
                    payload[
                        "train"
                    ]["date"].max()
                ),
                "CV nMAE mean (%)": (
                    optimization_run
                    .data.metrics.get(
                        "best_cv_nmae_pct_mean"
                    )
                ),
                "CV nMAE std (%)": (
                    optimization_run
                    .data.metrics.get(
                        "best_cv_nmae_pct_std"
                    )
                ),
                "OOT MAE (MW)": (
                    training_run
                    .data.metrics.get(
                        "oot_mae_mw"
                    )
                ),
                "OOT nMAE (%)": (
                    training_run
                    .data.metrics.get(
                        "oot_nmae_pct"
                    )
                ),
                "OOT MAE FC (%)": (
                    training_run
                    .data.metrics.get(
                        "oot_mae_fc_pct"
                    )
                ),
                "OOT R²": (
                    training_run
                    .data.metrics.get(
                        "oot_r2_score"
                    )
                ),
                "Peso LGBM": (
                    meta_weights.get(
                        "lgbm"
                    )
                ),
                "Peso XGB": (
                    meta_weights.get(
                        "xgboost"
                    )
                ),
                "Peso RF": (
                    meta_weights.get(
                        "rf"
                    )
                ),
                "Optuna study (min)": (
                    FROZEN_OPTUNA_STUDY_MINUTES[
                        label
                    ]
                ),
                "Training run (min)": (
                    run_duration_minutes(
                        training_run
                    )
                ),
                "Optimization Run ID": (
                    run_ids[
                        "optimization_run_id"
                    ]
                ),
                "Training Run ID": (
                    run_ids[
                        "training_run_id"
                    ]
                ),
            }
        )

    benchmark_results = (
        pd.DataFrame(
            benchmark_rows
        )
        .set_index("Janela")
    )

    display(
        benchmark_results.round(4)
    )


/tmp/ipykernel_90381/169441005.py:203: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  benchmark_results.round(4)


,Train rows,Train início,Train fim,CV nMAE mean (%),CV nMAE std (%),OOT MAE (MW),OOT nMAE (%),OOT MAE FC (%),OOT R²,Peso LGBM,Peso XGB,Peso RF,Optuna study (min),Training run (min),Optimization Run ID,Training Run ID
Janela,,,,,,,,,,,,,,,,
Expanding,21425,2024-03-21 00:00:00,2026-09-01 02:00:00,8.6949,0.6791,787.1836,6.6788,6.6788,0.8614,0.5890,0.3809,0.0000,16.1880,0.3276,2fdff0b534104041a01276c0a2fb6009,1d13a61244c54f06aa70f43a9993ea37
Rolling 24m,17486,2024-09-01 03:00:00,2026-09-01 02:00:00,9.1920,1.6572,937.1376,7.9511,7.9511,0.8217,0.0769,0.2916,0.5692,16.9885,0.2691,b145a449eb6245d4809ff9d3cc3ffc3c,a03c85cae21c4d4ebf0fd717d248b775
Rolling 12m,8760,2025-09-01 03:00:00,2026-09-01 02:00:00,8.1086,0.8133,889.6619,7.5483,7.5483,0.8297,0.5177,0.4331,0.0000,18.8824,0.4709,6bc964c478be44ca9e4a4cab2d071d08,9ce134ad54284ef9aee2cf29db69feb1


## 12. Diferenças em relação à expanding window

A expanding window é utilizada como baseline porque representa a
política anterior do projeto.

As diferenças abaixo são descritivas:

```text
Δ MAE  > 0  → maior erro que expanding
Δ nMAE > 0  → maior erro normalizado que expanding
Δ R²   < 0  → menor R² que expanding
```

Também calculamos a redução relativa do volume de treinamento.

Essas diferenças não constituem, por si só, teste de equivalência,
não-inferioridade ou significância estatística.


In [32]:
if benchmark_results is None:
    window_deltas = None

    print(
        "ℹ️ Execute ou congele as runs "
        "antes de calcular os deltas."
    )

else:
    baseline = (
        benchmark_results.loc[
            "Expanding"
        ]
    )

    window_deltas = (
        benchmark_results.loc[
            [
                "Rolling 24m",
                "Rolling 12m",
            ],
            [
                "Train rows",
                "CV nMAE mean (%)",
                "CV nMAE std (%)",
                "OOT MAE (MW)",
                "OOT nMAE (%)",
                "OOT R²",
                "Optuna study (min)",
                "Training run (min)",
            ],
        ]
        .copy()
    )

    window_deltas[
        "Redução TRAIN vs Expanding (%)"
    ] = (
        1
        - (
            window_deltas["Train rows"]
            / baseline["Train rows"]
        )
    ) * 100

    window_deltas[
        "Δ MAE vs Expanding (MW)"
    ] = (
        window_deltas[
            "OOT MAE (MW)"
        ]
        - baseline[
            "OOT MAE (MW)"
        ]
    )

    window_deltas[
        "Δ MAE vs Expanding (%)"
    ] = (
        window_deltas[
            "Δ MAE vs Expanding (MW)"
        ]
        / baseline[
            "OOT MAE (MW)"
        ]
    ) * 100

    window_deltas[
        "Δ nMAE vs Expanding (p.p.)"
    ] = (
        window_deltas[
            "OOT nMAE (%)"
        ]
        - baseline[
            "OOT nMAE (%)"
        ]
    )

    window_deltas[
        "Δ R² vs Expanding"
    ] = (
        window_deltas[
            "OOT R²"
        ]
        - baseline[
            "OOT R²"
        ]
    )

    display(
        window_deltas.round(4)
    )


,Train rows,CV nMAE mean (%),CV nMAE std (%),OOT MAE (MW),OOT nMAE (%),OOT R²,Optuna study (min),Training run (min),Redução TRAIN vs Expanding (%),Δ MAE vs Expanding (MW),Δ MAE vs Expanding (%),Δ nMAE vs Expanding (p.p.),Δ R² vs Expanding
Janela,,,,,,,,,,,,,
Rolling 24m,17486,9.1920,1.6572,937.1376,7.9511,0.8217,16.9885,0.2691,18.3851,149.9540,19.0494,1.2723,-0.0397
Rolling 12m,8760,8.1086,0.8133,889.6619,7.5483,0.8297,18.8824,0.4709,59.1132,102.4783,13.0183,0.8695,-0.0317


## 13. Interpretação do benchmark

### 13.1 Generalização no OOT

A expanding window apresentou o menor erro no holdout de
setembro/2026:

| Janela | CV nMAE mean (%) | CV nMAE std (%) | OOT MAE (MW) | OOT nMAE (%) | OOT R² |
|---|---:|---:|---:|---:|---:|
| Expanding | 8.6949 | 0.6791 | 787.1836 | 6.6788 | 0.8614 |
| Rolling 24m | 9.1920 | 1.6572 | 937.1376 | 7.9511 | 0.8217 |
| Rolling 12m | 8.1086 | 0.8133 | 889.6619 | 7.5483 | 0.8297 |

Em relação à expanding window:

- Rolling 24m aumentou o MAE em aproximadamente **149.95 MW**
  (**+19.05%**) e o nMAE em cerca de **+1.27 p.p.**;
- Rolling 12m aumentou o MAE em aproximadamente **102.48 MW**
  (**+13.02%**) e o nMAE em cerca de **+0.87 p.p.**.

Portanto, neste snapshot e neste OOT, reduzir o histórico para 12 ou
24 meses não melhorou a generalização.

### 13.2 CV interno versus futuro realmente não visto

O Rolling 12m produziu o melhor CV nMAE médio (**8.1086%**), mas não o
melhor OOT.

Esse resultado mostra uma diferença relevante entre:

```text
validação temporal dentro do histórico recente
                    ≠
generalização no mês futuro congelado
```

Não é necessário interpretar isso automaticamente como overfitting.
O que o benchmark demonstra é uma **discordância entre o desempenho
durante a validação temporal interna e o desempenho no holdout futuro**.

Esse é precisamente o motivo para manter um OOT temporal independente.

### 13.3 Estabilidade dos splits temporais

A dispersão do CV também mudou com a janela:

```text
Expanding     std = 0.6791
Rolling 24m   std = 1.6572
Rolling 12m   std = 0.8133
```

O Rolling 24m apresentou simultaneamente:

- maior CV nMAE médio;
- maior variabilidade entre splits;
- pior MAE/nMAE no OOT;
- menor R² no OOT.

Assim, não há sinal favorável à janela de 24 meses neste benchmark.

### 13.4 Composição do ensemble

Os coeficientes OOF do meta-learner foram:

| Janela | LGBM | XGBoost | Random Forest |
|---|---:|---:|---:|
| Expanding | 0.5890 | 0.3809 | 0.0000 |
| Rolling 24m | 0.0769 | 0.2916 | 0.5692 |
| Rolling 12m | 0.5177 | 0.4331 | 0.0000 |

Os coeficientes são positivos, mas **não são percentuais normalizados**,
pois o meta-learner utiliza `LinearRegression(positive=True,
fit_intercept=False)` sem restrição de soma igual a 1.

O Random Forest recebeu coeficiente zero na expanding e no Rolling 12m,
mas tornou-se o maior coeficiente no Rolling 24m. Isso mostra que a
contribuição relativa dos base learners depende da janela temporal.

Esse resultado não demonstra que o Random Forest seja universalmente
necessário, mas valida a decisão experimental de manter
`LGBM + XGBoost + Random Forest` fixos durante o benchmark. Remover um
estimador antes desta comparação teria introduzido uma segunda variável
experimental.

### 13.5 Volume de dados e custo computacional

A redução do TRAIN foi substancial:

```text
Expanding     21,425 linhas
Rolling 24m   17,486 linhas
Rolling 12m    8,760 linhas
```

Apesar disso, o tempo do estudo Optuna não caiu monotonicamente com o
número de linhas. A duração correta observada foi aproximadamente:

```text
Expanding      16.19 min
Rolling 24m    16.99 min
Rolling 12m    18.88 min
```

Isso é plausível porque o custo de cada trial também depende dos
hiperparâmetros amostrados — número de árvores, profundidade e demais
parâmetros — e não apenas do número de amostras.

Por isso, este benchmark não fornece evidência de que uma rolling
window menor gere automaticamente economia computacional.

### 13.6 O que este benchmark não demonstra

Os resultados não justificam afirmar que:

- mais histórico será sempre melhor;
- expanding é universalmente superior;
- Rolling 12m e Rolling 24m são estatisticamente inferiores em qualquer
  período futuro;
- Random Forest é sempre necessário;
- as diferenças observadas constituem um teste formal de significância,
  equivalência ou não-inferioridade.

A conclusão vale para o snapshot congelado, a arquitetura fixa e o OOT
de setembro/2026 utilizados neste experimento.


## 14. Quality Gate e isolamento operacional

Este benchmark foi deliberadamente separado da promoção automática.

O Quality Gate atual do projeto avalia Champion vs Challenger no
`continuous_training_pipeline()`. Esse mecanismo não era apropriado
para executar três tratamentos experimentais em sequência, pois cada
run poderia produzir uma decisão operacional antes de a comparação
estar concluída.

Neste notebook:

```text
Quality Gate executado: NÃO
Champion alterado automaticamente: NÃO
Alias @champion alterado: NÃO
```

As versões experimentais registradas no Model Registry permanecem como
evidência técnica, sem promoção automática.

Uma eventual mudança da política de janela deve ser incorporada ao
fluxo operacional em commit separado e submetida novamente aos testes e
ao CI/CD.


## 15. Conclusão

Com o snapshot atual e o OOT congelado de setembro/2026, a
**expanding window apresentou a melhor generalização entre as três
políticas avaliadas**.

A expanding window obteve:

```text
TRAIN rows: 21,425
CV nMAE mean: 8.6949%
CV nMAE std: 0.6791%
OOT MAE: 787.18 MW
OOT nMAE: 6.6788%
OOT R²: 0.8614
```

O Rolling 24m reduziu o histórico para 17,486 observações, mas apresentou
pior desempenho tanto no CV quanto no OOT, além da maior variabilidade
entre os splits temporais.

O Rolling 12m reduziu o histórico para 8,760 observações e apresentou o
melhor CV nMAE médio do benchmark. Entretanto, essa vantagem não se
transferiu para o OOT de setembro/2026: seu MAE ficou aproximadamente
102.48 MW acima da expanding window.

Portanto, o principal resultado não é que **mais dados são sempre
melhores**. O resultado é mais específico:

> Para este snapshot, esta arquitetura e este holdout futuro, o histórico
> adicional preservado pela expanding window trouxe melhor generalização
> do que restringir o treinamento aos últimos 12 ou 24 meses.

A política da janela temporal foi tornada configurável justamente para evitar
transformar essa evidência em uma regra permanente. Mudanças futuras de
regime podem alterar a relação entre histórico antigo e desempenho, e o
benchmark pode ser repetido com novos snapshots/OOTs.

O experimento também mostrou que a composição do ensemble é sensível à
janela temporal. O Random Forest recebeu coeficiente zero em duas
janelas, mas tornou-se dominante no Rolling 24m. Isso reforça que manter
os três estimadores fixos foi importante para isolar corretamente a
janela de treinamento como única variável experimental.

### Decisão experimental

```text
Expanding:
melhor generalização no OOT e menor variabilidade de CV.

Rolling 24m:
não suportado pelos resultados atuais.

Rolling 12m:
CV interno favorável, mas sem ganho no OOT.

Política candidata para validação operacional:
manter expanding como política de referência no snapshot atual.
```

Essa decisão é experimental e contextual, não uma afirmação de
superioridade universal da expanding window.


## 16. Próximos passos

O benchmark de janelas está concluído. Os próximos passos são:

1. manter os três Run IDs finais congelados neste notebook;
2. manter `RUN_EXPERIMENTS=False`;
3. registrar a conclusão no notebook técnico principal e no Model Card;
4. manter a expanding window como política de referência no snapshot
   atual;
5. não introduzir uma regra automática do tipo “expanding sempre vence”;
6. repetir o benchmark apenas quando houver nova evidência que justifique
   reavaliar a política temporal — por exemplo, novo snapshot, novo OOT
   ou degradação observada em produção;
7. tratar monitoramento de drift como mecanismo de detecção e evidência,
   não como substituto para validação temporal experimental.

Qualquer alteração futura na política operacional deve ser feita em
commit separado e passar novamente por testes, `tox` e CI/CD.
